# LoRA finetuning a released AstroPT checkpoint

Loads a real pretrained checkpoint from
[`Smith42/astroPT`](https://huggingface.co/Smith42/astroPT) (Smith et al.,
2024, https://arxiv.org/abs/2405.14930) and LoRA-finetunes it (Hu et al.,
2021, https://arxiv.org/abs/2106.09685) for galaxy morphology classification
on [`UniverseTBD/mmu_gz10`](https://huggingface.co/datasets/UniverseTBD/mmu_gz10)
(Galaxy10 DECals, 17,736 galaxies, 10 classes), following the reference
model's own `scripts/finetune.py` recipe: freeze the pretrained weights,
train only injected low-rank adapters and a new task head.

This uses the smallest released checkpoint (1M parameters) for tutorial
speed; larger ones (5M to 2.1B) are listed at the same repo under
`models/fully_trained/`.

**Checkpoint compatibility.** The released checkpoints predate this
library's simplified, single-modality reimplementation, so only part of the
checkpoint loads — see `load_pretrained_backbone` below for what transfers
and why.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [ ]:
!pip install -q datasets torchvision scikit-learn huggingface_hub

## Imports

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from huggingface_hub import hf_hub_download

import astrolens
from astrolens.models.astropt import AstroPT
from utils import gz10

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the dataset and split 70/10/20

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
CLASS_NAMES = gz10.CLASS_NAMES
NUM_CLASSES = gz10.NUM_CLASSES

data, labels, train_idx, val_idx, test_idx = gz10.load_split_702010()

train_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomRotation(90),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(gz10.IMAGE_MEAN, gz10.IMAGE_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(gz10.IMAGE_MEAN, gz10.IMAGE_STD),
    ]
)

train_dataset = gz10.GZ10Dataset(data, train_idx, train_transform)
val_dataset = gz10.GZ10Dataset(data, val_idx, eval_transform)
test_dataset = gz10.GZ10Dataset(data, test_idx, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)

len(train_dataset), len(val_dataset), len(test_dataset)

## Download and load the pretrained backbone

`Smith42/astroPT` checkpoints predate this library's multimodal-free
reimplementation: they store the causal transformer under `transformer.h.*`
(compiled with `torch.compile`, hence the `_orig_mod.` prefix to strip) with
a patch encoder/decoder (`transformer.wte.*` / `lm_head.*`) too deep to line
up with `AstroPT`'s single-layer patch projection. `load_pretrained_backbone`
transfers everything that does line up exactly — attention, MLP, layer
norms, and the learned position embeddings (sliced to how many patches this
image size needs) — and leaves the encoder, decoder, LoRA adapters, and
classification head freshly initialized, to be learned during finetuning.

In [ ]:
def load_pretrained_backbone(model: AstroPT, repo_id: str, filename: str) -> dict:
    """Load the transformer body of a released AstroPT checkpoint into `model`.

    Returns the checkpoint's `model_args` dict (n_layer, n_head, n_embd,
    patch_size, block_size, ...) so the caller can verify `model` was
    constructed with matching dimensions.
    """
    path = hf_hub_download(repo_id=repo_id, filename=filename)
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    state_dict = {k.replace("_orig_mod.", ""): v for k, v in checkpoint["model"].items()}
    model_args = checkpoint["model_args"]

    num_patches = model.grid[0] * model.grid[1]
    mapped = {
        "pos_embed.weight": state_dict["transformer.wpe.weight"][:num_patches],
        "ln_f.weight": state_dict["transformer.ln_f.weight"],
    }
    for i in range(model_args["n_layer"]):
        src, dst = f"transformer.h.{i}.", f"blocks.{i}."
        mapped[dst + "ln_1.weight"] = state_dict[src + "ln_1.weight"]
        mapped[dst + "attn.qkv.weight"] = state_dict[src + "attn.c_attn.weight"]
        mapped[dst + "attn.proj.weight"] = state_dict[src + "attn.c_proj.weight"]
        mapped[dst + "ln_2.weight"] = state_dict[src + "ln_2.weight"]
        mapped[dst + "mlp.0.weight"] = state_dict[src + "mlp.c_fc.weight"]
        mapped[dst + "mlp.2.weight"] = state_dict[src + "mlp.c_proj.weight"]

    missing, unexpected = model.load_state_dict(mapped, strict=False)
    assert not unexpected, f"unexpected keys, checkpoint format may have changed: {unexpected}"
    return model_args


REPO_ID = "Smith42/astroPT"
CKPT_FILENAME = "models/fully_trained/0001M_params/030000_ckpt.pt"
LORA_R = 8

# a throwaway probe to read model_args before building the real model
_probe_path = hf_hub_download(repo_id=REPO_ID, filename=CKPT_FILENAME)
_model_args = torch.load(_probe_path, map_location="cpu", weights_only=False)["model_args"]

model = astrolens.create_model(
    "astropt",
    img_size=IMG_SIZE,
    in_chans=_model_args["n_chan"],
    patch_size=_model_args["patch_size"],
    dim=_model_args["n_embd"],
    depth=_model_args["n_layer"],
    heads=_model_args["n_head"],
    num_classes=NUM_CLASSES,
    lora_r=LORA_R,
    spiral=True,  # this checkpoint was pretrained with spiral patch ordering
).to(device)

load_pretrained_backbone(model, REPO_ID, CKPT_FILENAME)
model.mark_only_lora_as_trainable()

sum(p.numel() for p in model.parameters() if p.requires_grad)

## Finetune

In [ ]:
FINETUNE_EPOCHS = 20
FINETUNE_LR = 1e-3

# inverse-frequency class weights from the train split, following the
# Linformer notebook's approach to counter GZ10's class imbalance
criterion = nn.CrossEntropyLoss(weight=gz10.class_weights(labels, train_idx).to(device))
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=FINETUNE_LR
)

for epoch in range(1, FINETUNE_EPOCHS + 1):
    train_loss, train_acc = gz10.run_classification_epoch(
        model, train_loader, criterion, device, train=True, optimizer=optimizer
    )
    val_loss, val_acc = gz10.run_classification_epoch(model, val_loader, criterion, device, train=False)
    print(
        f"epoch {epoch:02d} train_loss={train_loss:.4f} train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
    )

## Evaluate on the held-out test split

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels_batch in test_loader:
        logits = model(images.to(device))
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())
        y_true.extend(labels_batch.tolist())

test_acc = accuracy_score(y_true, y_pred)
test_f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"test_acc={test_acc:.3f} test_f1_macro={test_f1_macro:.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))